In [55]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType, DoubleType
from pyspark.sql.functions import count,when,col,sum,round
from pyspark.sql import functions as F
from pyspark.sql.window import Window   

In [56]:
spark=SparkSession.builder.appName("Schema").getOrCreate()
schema = StructType([
    StructField("event_id",   LongType(),      True),
    StructField("event_type", StringType(),    True),
    StructField("ts",         TimestampType(), True),
    StructField("amount",     DoubleType(),    True),
])

In [57]:
df=spark.read.schema(schema).option("mode", "FAILFAST").json("events.json")
df.printSchema()
df.show(5,truncate=False)

root
 |-- event_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- amount: double (nullable = true)

+--------+----------+-------------------+------+
|event_id|event_type|ts                 |amount|
+--------+----------+-------------------+------+
|1       |view      |2024-09-07 02:15:35|471.3 |
|2       |signup    |2024-05-22 20:48:29|74.85 |
|3       |signup    |2024-02-05 11:32:29|287.3 |
|4       |view      |2024-04-10 17:38:23|246.22|
|5       |signup    |2024-06-07 07:12:27|97.4  |
+--------+----------+-------------------+------+
only showing top 5 rows


In [58]:
df=df.dropna(subset=["event_id","event_type","ts"])
dq_summary = df.select([
    count("*").alias("total_rows"),
    count(when(col("event_id").isNull(), True)).alias("event_id_nulls"),
    count(when(col("event_type").isNull(), True)).alias("event_type_nulls"),
    count(when(col("ts").isNull(), True)).alias("ts_nulls"),
])

dq_summary.show()

+----------+--------------+----------------+--------+
|total_rows|event_id_nulls|event_type_nulls|ts_nulls|
+----------+--------------+----------------+--------+
|      1000|             0|               0|       0|
+----------+--------------+----------------+--------+



In [62]:
event_window_running = Window.partitionBy("event_type").orderBy("ts")
event_window=Window.partitionBy("event_type")
df = df.withColumn("event_type_count", F.count("event_type").over(event_window))

df_rank = df.select("event_type","event_type_count").distinct()
rank_window = Window.orderBy(col("event_type_count").desc())
df_rank = df_rank.withColumn("rank", F.rank().over(rank_window))
df_rank.show(5,truncate=False)
df=df.withColumn("event_type_rank", F.rank().over(rank_window))
df.show(5,truncate=False)

+----------+----------------+----+
|event_type|event_type_count|rank|
+----------+----------------+----+
|signup    |267             |1   |
|click     |256             |2   |
|view      |242             |3   |
|purchase  |235             |4   |
+----------+----------------+----+

+--------+----------+-------------------+------+----------------+---------------+
|event_id|event_type|ts                 |amount|event_type_count|event_type_rank|
+--------+----------+-------------------+------+----------------+---------------+
|2       |signup    |2024-05-22 20:48:29|74.85 |267             |1              |
|12      |signup    |2024-12-09 07:16:11|27.85 |267             |1              |
|3       |signup    |2024-02-05 11:32:29|287.3 |267             |1              |
|5       |signup    |2024-06-07 07:12:27|97.4  |267             |1              |
|7       |signup    |2024-03-28 23:56:21|65.74 |267             |1              |
+--------+----------+-------------------+------+---------------